# Taller 9: Extensiones No Lineales

En este taller exploraremos cómo extender los modelos lineales a formas no lineales mediante interacciones y términos polinómicos. Analizaremos cómo implementar estos modelos, interpretar sus resultados y manejar los problemas de multicolinealidad que suelen surgir.

## Objetivos de Aprendizaje
- Implementar y evaluar términos de interacción entre variables
- Incorporar términos polinómicos en modelos lineales
- Gestionar la multicolinealidad inducida por las transformaciones no lineales
- Validar la complejidad del modelo mediante técnicas apropiadas
- Interpretar correctamente los coeficientes en modelos con términos no lineales


In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings

# Configuración para visualizaciones
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')
warnings.filterwarnings('ignore')
%matplotlib inline


# Fundamentos Teóricos de Extensiones No Lineales

## 1. ¿Qué son las Extensiones No Lineales?

Los modelos lineales estándar asumen una relación lineal entre predictores y la variable respuesta:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_p x_p + \varepsilon$$

Sin embargo, muchos fenómenos del mundo real exhiben relaciones no lineales. Las extensiones no lineales nos permiten capturar estas relaciones mientras seguimos utilizando el marco de los modelos lineales.

## 2. Principales Tipos de Extensiones No Lineales

### 2.1 Términos de Interacción

Las interacciones modelan el efecto conjunto de dos variables, donde el efecto de una variable depende del valor de otra:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 (x_1 \times x_2) + \varepsilon$$

Aquí, $\beta_3$ captura cómo el efecto de $x_1$ sobre $y$ varía según el valor de $x_2$ (y viceversa).

### 2.2 Términos Polinómicos

Los términos polinómicos capturan relaciones curvilíneas entre una variable predictora y la respuesta:

$$y = \beta_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3 + \ldots + \varepsilon$$

Un término cuadrático ($x^2$) modela una parábola, un término cúbico ($x^3$) permite un punto de inflexión adicional, etc.

### 2.3 Transformaciones No Lineales

También podemos aplicar transformaciones no lineales a las variables:

$$y = \beta_0 + \beta_1 \log(x_1) + \beta_2 \sqrt{x_2} + \varepsilon$$

Estas transformaciones pueden linearizar relaciones intrínsecamente no lineales.

## 3. Consideraciones Importantes

### 3.1 Multicolinealidad Inducida

La inclusión de términos no lineales suele introducir multicolinealidad, ya que estos términos están correlacionados con los términos lineales originales. Esto puede:
- Inflar la varianza de los estimadores
- Hacer que los coeficientes sean inestables
- Complicar la interpretación

### 3.2 Interpretación de Coeficientes

La interpretación de los coeficientes se vuelve más compleja:
- En modelos con interacciones, el efecto de una variable depende del valor de otra
- En modelos polinómicos, el efecto marginal varía en función del valor de la variable

### 3.3 Sobreajuste (Overfitting)

Los modelos no lineales tienen mayor capacidad expresiva, lo que aumenta el riesgo de sobreajuste. Es fundamental validar la complejidad del modelo mediante:
- Validación cruzada
- Criterios de información (AIC, BIC)
- Análisis de residuos


# Implementación de Funciones para Términos No Lineales

A continuación, definiremos funciones para implementar modelos con términos de interacción y polinómicos, así como para evaluar y visualizar sus resultados.


In [ ]:

def crear_interacciones(X, grado=2):
    """
    Crea términos de interacción entre las variables de X.
    
    Parámetros:
    -----------
    X : DataFrame
        Variables predictoras originales
    grado : int, opcional (default=2)
        Grado máximo de las interacciones (2=dos vías, 3=tres vías, etc.)
        
    Retorna:
    --------
    X_inter : DataFrame
        DataFrame con las variables originales y las interacciones
    """
    if isinstance(X, pd.DataFrame):
        poly = PolynomialFeatures(degree=grado, include_bias=False, interaction_only=True)
        X_poly_array = poly.fit_transform(X)
        
        # Obtener los nombres de las características
        nombres_orig = X.columns
        nombres_inter = []
        
        # Generar nombres para los términos de interacción
        for i, indices in enumerate(poly.powers_):
            if np.sum(indices) <= 1:  # Variable original
                if np.sum(indices) == 1:
                    idx = np.where(indices == 1)[0][0]
                    nombres_inter.append(nombres_orig[idx])
            else:  # Término de interacción
                nombre = '*'.join([f"{nombres_orig[j]}" for j, exp in enumerate(indices) if exp > 0])
                nombres_inter.append(nombre)
        
        # Crear DataFrame con las interacciones
        X_inter = pd.DataFrame(X_poly_array, columns=nombres_inter, index=X.index)
        
        return X_inter
    else:
        # Si X no es un DataFrame, convertirlo
        X_df = pd.DataFrame(X)
        return crear_interacciones(X_df, grado)

def crear_polinomios(X, grado=2, interacciones=True):
    """
    Crea términos polinómicos para las variables de X.
    
    Parámetros:
    -----------
    X : DataFrame
        Variables predictoras originales
    grado : int, opcional (default=2)
        Grado máximo de los polinomios
    interacciones : bool, opcional (default=True)
        Si True, incluye términos de interacción
        
    Retorna:
    --------
    X_poly : DataFrame
        DataFrame con las variables originales y los términos polinómicos
    """
    if isinstance(X, pd.DataFrame):
        poly = PolynomialFeatures(degree=grado, include_bias=False, interaction_only=not interacciones)
        X_poly_array = poly.fit_transform(X)
        
        # Obtener los nombres de las características
        nombres_orig = X.columns
        nombres_poly = []
        
        # Generar nombres para los términos polinómicos
        for i, indices in enumerate(poly.powers_):
            if np.sum(indices) == 0:  # Término constante (bias)
                continue
            elif np.sum(indices) == 1:  # Variable original
                idx = np.where(indices == 1)[0][0]
                nombres_poly.append(nombres_orig[idx])
            else:  # Término polinómico
                nombre = '*'.join([f"{nombres_orig[j]}^{exp}" if exp > 1 else f"{nombres_orig[j]}" 
                                  for j, exp in enumerate(indices) if exp > 0])
                nombres_poly.append(nombre)
        
        # Crear DataFrame con los términos polinómicos
        X_poly = pd.DataFrame(X_poly_array, columns=nombres_poly, index=X.index)
        
        return X_poly
    else:
        # Si X no es un DataFrame, convertirlo
        X_df = pd.DataFrame(X)
        return crear_polinomios(X_df, grado, interacciones)

def calcular_vif(X):
    """
    Calcula los factores de inflación de la varianza (VIF) para detectar multicolinealidad.
    
    Parámetros:
    -----------
    X : DataFrame
        Variables predictoras
        
    Retorna:
    --------
    vif_df : DataFrame
        DataFrame con los VIF de cada variable
    """
    # Añadir constante para el intercepto
    X_sm = sm.add_constant(X)
    
    # Calcular VIF para cada predictor
    vif_data = pd.DataFrame()
    vif_data["Variable"] = X_sm.columns
    vif_data["VIF"] = [variance_inflation_factor(X_sm.values, i) for i in range(X_sm.shape[1])]
    
    return vif_data

def evaluar_modelo_cv(X, y, grado=2, interacciones=True, cv=5, metrica='neg_mean_squared_error'):
    """
    Evalúa un modelo polinómico usando validación cruzada.
    
    Parámetros:
    -----------
    X : DataFrame o array
        Variables predictoras originales
    y : Series o array
        Variable respuesta
    grado : int, opcional (default=2)
        Grado máximo de los polinomios
    interacciones : bool, opcional (default=True)
        Si True, incluye términos de interacción
    cv : int, opcional (default=5)
        Número de folds para validación cruzada
    metrica : str, opcional (default='neg_mean_squared_error')
        Métrica de evaluación
        
    Retorna:
    --------
    resultados : dict
        Diccionario con los resultados de la evaluación
    """
    # Crear pipeline con escalado y modelo
    pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=grado, include_bias=False, interaction_only=not interacciones)),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ])
    
    # Realizar validación cruzada
    cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring=metrica)
    
    # Para métricas negativas, convertir a positivas
    if metrica.startswith('neg_'):
        cv_scores = -cv_scores
    
    # Ajustar modelo en todos los datos para referencia
    pipeline.fit(X, y)
    y_pred = pipeline.predict(X)
    r2 = r2_score(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    
    # Guardar resultados
    resultados = {
        'grado': grado,
        'interacciones': interacciones,
        'cv_scores': cv_scores,
        'cv_mean': np.mean(cv_scores),
        'cv_std': np.std(cv_scores),
        'r2_train': r2,
        'mse_train': mse
    }
    
    return resultados

def visualizar_efectos_no_lineales(modelo, X, y, var_interes, var_interaccion=None, n_puntos=100):
    """
    Visualiza los efectos no lineales en un modelo con términos polinómicos o interacciones.
    
    Parámetros:
    -----------
    modelo : objeto de modelo ajustado
        Modelo de regresión ajustado
    X : DataFrame
        Variables predictoras utilizadas para ajustar el modelo
    y : Series o array
        Variable respuesta
    var_interes : str
        Nombre de la variable de interés para visualizar
    var_interaccion : str, opcional
        Nombre de la variable con la que interactúa (para mostrar interacciones)
    n_puntos : int, opcional (default=100)
        Número de puntos para la visualización
        
    Retorna:
    --------
    fig : objeto Figure de matplotlib
        Figura con la visualización
    """
    # Crear rango de valores para la variable de interés
    x_min, x_max = X[var_interes].min(), X[var_interes].max()
    x_range = np.linspace(x_min, x_max, n_puntos)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Graficar los datos originales
    ax.scatter(X[var_interes], y, alpha=0.4, label='Datos')
    
    # Caso 1: Sin interacción especificada
    if var_interaccion is None:
        # Crear datos para predicción
        X_pred = X.copy()
        y_preds = []
        
        for x_val in x_range:
            X_pred[var_interes] = x_val
            # Recrear las características polinómicas o de interacción si es necesario
            if hasattr(modelo, 'named_steps') and 'poly' in modelo.named_steps:
                X_transformed = modelo.named_steps['poly'].transform(X_pred)
                if 'scaler' in modelo.named_steps:
                    X_transformed = modelo.named_steps['scaler'].transform(X_transformed)
                y_pred = modelo.named_steps['model'].predict(X_transformed)
            else:
                y_pred = modelo.predict(X_pred)
            y_preds.append(np.mean(y_pred))
        
        # Graficar la curva de predicción
        ax.plot(x_range, y_preds, 'r-', linewidth=2, label='Predicción del modelo')
        
    # Caso 2: Con interacción
    else:
        # Obtener valores para la variable de interacción
        inter_q25 = X[var_interaccion].quantile(0.25)
        inter_q50 = X[var_interaccion].quantile(0.50)
        inter_q75 = X[var_interaccion].quantile(0.75)
        
        inter_values = [inter_q25, inter_q50, inter_q75]
        labels = [f"{var_interaccion} (Q1)", f"{var_interaccion} (Q2)", f"{var_interaccion} (Q3)"]
        colors = ['b', 'g', 'orange']
        
        for i, inter_val in enumerate(inter_values):
            X_pred = X.copy()
            X_pred[var_interaccion] = inter_val
            y_preds = []
            
            for x_val in x_range:
                X_pred[var_interes] = x_val
                # Recrear las características polinómicas o de interacción si es necesario
                if hasattr(modelo, 'named_steps') and 'poly' in modelo.named_steps:
                    X_transformed = modelo.named_steps['poly'].transform(X_pred)
                    if 'scaler' in modelo.named_steps:
                        X_transformed = modelo.named_steps['scaler'].transform(X_transformed)
                    y_pred = modelo.named_steps['model'].predict(X_transformed)
                else:
                    y_pred = modelo.predict(X_pred)
                y_preds.append(np.mean(y_pred))
            
            # Graficar la curva de predicción para este nivel de interacción
            ax.plot(x_range, y_preds, color=colors[i], linewidth=2, 
                    label=f'Predicción ({labels[i]}={inter_val:.2f})')
    
    ax.set_xlabel(var_interes)
    ax.set_ylabel('Variable Respuesta')
    
    if var_interaccion is None:
        ax.set_title(f'Efecto No Lineal de {var_interes}')
    else:
        ax.set_title(f'Interacción entre {var_interes} y {var_interaccion}')
    
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    return fig


# Generación de Datos para Ejemplos

A continuación, generaremos datos sintéticos con relaciones no lineales para ilustrar la implementación y análisis de modelos con términos polinómicos e interacciones.


In [ ]:

def generar_datos_no_lineales(n=200, seed=42):
    """
    Genera datos sintéticos con relaciones no lineales y efectos de interacción.
    
    Parámetros:
    -----------
    n : int, opcional (default=200)
        Número de observaciones
    seed : int, opcional (default=42)
        Semilla para reproducibilidad
        
    Retorna:
    --------
    X : DataFrame
        Variables predictoras
    y : Series
        Variable respuesta
    """
    np.random.seed(seed)
    
    # Generar predictores 
    X1 = np.random.uniform(-3, 3, n)
    X2 = np.random.uniform(-3, 3, n)
    
    # Generar ruido aleatorio
    error = np.random.normal(0, 1, n)
    
    # Crear variable respuesta con términos cuadráticos e interacción
    y = 2 + 1.5 * X1 - 2 * X2 + 1.2 * X1**2 - 0.8 * X2**2 + 2.5 * X1 * X2 + error
    
    # Convertir a DataFrame/Series para mejor manejo
    X = pd.DataFrame({'X1': X1, 'X2': X2})
    y = pd.Series(y, name='Y')
    
    return X, y

# Generar datos para nuestros ejemplos
X, y = generar_datos_no_lineales(n=200)

# Mostrar estadísticas descriptivas
print("Estadísticas descriptivas de las variables generadas:")
descripcion = pd.concat([X, y.to_frame()], axis=1).describe()
print(descripcion)

# Visualizar la distribución de las variables y relaciones
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(X['X1'], kde=True, ax=axes[0])
axes[0].set_title('Distribución de X1')

sns.histplot(X['X2'], kde=True, ax=axes[1])
axes[1].set_title('Distribución de X2')

sns.histplot(y, kde=True, ax=axes[2])
axes[2].set_title('Distribución de Y')

plt.tight_layout()
plt.show()

# Visualizar relaciones entre predictores y respuesta
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(x='X1', y='Y', data=pd.concat([X, y.to_frame()], axis=1), ax=axes[0])
axes[0].set_title('Relación X1 vs Y')

sns.scatterplot(x='X2', y='Y', data=pd.concat([X, y.to_frame()], axis=1), ax=axes[1])
axes[1].set_title('Relación X2 vs Y')

# Crear gráfico 3D para visualizar la relación conjunta
from mpl_toolkits.mplot3d import Axes3D
ax3d = fig.add_subplot(133, projection='3d')
ax3d.scatter(X['X1'], X['X2'], y, alpha=0.6)
ax3d.set_xlabel('X1')
ax3d.set_ylabel('X2')
ax3d.set_zlabel('Y')
ax3d.set_title('Relación 3D entre X1, X2 e Y')

plt.tight_layout()
plt.show()


# Implementación y Comparación de Modelos Lineales y No Lineales

A continuación, implementaremos diferentes modelos para comparar enfoques lineales y no lineales utilizando los datos generados.


In [ ]:

# Dividir datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("Dimensiones del conjunto de entrenamiento: X_train:", X_train.shape, "y_train:", y_train.shape)
print("Dimensiones del conjunto de prueba: X_test:", X_test.shape, "y_test:", y_test.shape)

# 1. Modelo Lineal Base (sin términos no lineales)
print("\n" + "="*50)
print("MODELO 1: REGRESIÓN LINEAL SIN TÉRMINOS NO LINEALES")
print("="*50)

modelo_lineal = sm.OLS(y_train, sm.add_constant(X_train)).fit()
print(modelo_lineal.summary())

# Evaluar en conjunto de prueba
X_test_const = sm.add_constant(X_test)
y_pred_lineal = modelo_lineal.predict(X_test_const)
mse_lineal = mean_squared_error(y_test, y_pred_lineal)
r2_lineal = r2_score(y_test, y_pred_lineal)

print(f"Métricas en conjunto de prueba:")
print(f"MSE: {mse_lineal:.4f}")
print(f"R²: {r2_lineal:.4f}")

# 2. Modelo con Términos de Interacción
print("\n" + "="*50)
print("MODELO 2: REGRESIÓN LINEAL CON TÉRMINOS DE INTERACCIÓN")
print("="*50)

# Crear matriz de interacciones
X_train_inter = crear_interacciones(X_train)
print("Variables en el modelo con interacciones:", X_train_inter.columns.tolist())

# Ajustar modelo
modelo_inter = sm.OLS(y_train, sm.add_constant(X_train_inter)).fit()
print(modelo_inter.summary())

# Evaluar en conjunto de prueba
X_test_inter = crear_interacciones(X_test)
X_test_inter_const = sm.add_constant(X_test_inter)
y_pred_inter = modelo_inter.predict(X_test_inter_const)
mse_inter = mean_squared_error(y_test, y_pred_inter)
r2_inter = r2_score(y_test, y_pred_inter)

print(f"Métricas en conjunto de prueba:")
print(f"MSE: {mse_inter:.4f}")
print(f"R²: {r2_inter:.4f}")

# Calcular VIF para el modelo con interacciones
print("\nFactores de Inflación de la Varianza (VIF) para el modelo con interacciones:")
vif_inter = calcular_vif(X_train_inter)
print(vif_inter.sort_values('VIF', ascending=False))

# 3. Modelo con Términos Polinómicos (grado 2)
print("\n" + "="*50)
print("MODELO 3: REGRESIÓN LINEAL CON TÉRMINOS POLINÓMICOS (GRADO 2)")
print("="*50)

# Crear matriz con términos polinómicos
X_train_poly2 = crear_polinomios(X_train, grado=2)
print("Variables en el modelo polinómico de grado 2:", X_train_poly2.columns.tolist())

# Ajustar modelo
modelo_poly2 = sm.OLS(y_train, sm.add_constant(X_train_poly2)).fit()
print(modelo_poly2.summary())

# Evaluar en conjunto de prueba
X_test_poly2 = crear_polinomios(X_test, grado=2)
X_test_poly2_const = sm.add_constant(X_test_poly2)
y_pred_poly2 = modelo_poly2.predict(X_test_poly2_const)
mse_poly2 = mean_squared_error(y_test, y_pred_poly2)
r2_poly2 = r2_score(y_test, y_pred_poly2)

print(f"Métricas en conjunto de prueba:")
print(f"MSE: {mse_poly2:.4f}")
print(f"R²: {r2_poly2:.4f}")

# Calcular VIF para el modelo polinómico
print("\nFactores de Inflación de la Varianza (VIF) para el modelo polinómico:")
vif_poly = calcular_vif(X_train_poly2)
print(vif_poly.sort_values('VIF', ascending=False))


# Visualización e Interpretación de Efectos No Lineales

A continuación, visualizaremos los efectos no lineales y las interacciones en nuestros modelos para facilitar su interpretación.


In [ ]:

# Crear modelo con scikit-learn para poder usar la función de visualización
# Modelo polinómico
pipeline_poly = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])
pipeline_poly.fit(X_train, y_train)

# 1. Visualizar efecto cuadrático de X1
print("Visualización del efecto cuadrático de X1:")
fig_x1 = visualizar_efectos_no_lineales(pipeline_poly, X, y, 'X1')
plt.show()

# 2. Visualizar efecto cuadrático de X2
print("Visualización del efecto cuadrático de X2:")
fig_x2 = visualizar_efectos_no_lineales(pipeline_poly, X, y, 'X2')
plt.show()

# 3. Visualizar interacción entre X1 y X2
print("Visualización de la interacción entre X1 y X2:")
fig_inter = visualizar_efectos_no_lineales(pipeline_poly, X, y, 'X1', 'X2')
plt.show()

# Interpretación de los coeficientes
print("\n" + "="*50)
print("INTERPRETACIÓN DE COEFICIENTES EN MODELOS NO LINEALES")
print("="*50)

# Para el modelo polinómico
coefs_poly = dict(zip(["const"] + X_train_poly2.columns.tolist(), modelo_poly2.params))
print("Coeficientes del modelo polinómico:")
for nombre, valor in coefs_poly.items():
    print(f"{nombre}: {valor:.4f}")

# Explicar cómo calcular el efecto marginal
print("\nEfecto Marginal en Modelos No Lineales:")
print("En un modelo con términos cuadráticos e interacciones, el efecto marginal varía según los valores de las variables.")

# Calcular y visualizar el efecto marginal de X1 para diferentes valores de X2
X1_values = np.linspace(X['X1'].min(), X['X1'].max(), 5)
X2_values = np.percentile(X['X2'], [25, 50, 75])  # Q1, Q2, Q3

print("\nEfecto Marginal de X1 para diferentes valores de X2:")
print("X1".center(10) + "X2 (Q1)".center(15) + "X2 (Q2)".center(15) + "X2 (Q3)".center(15))
print("-" * 55)

# Extraer coeficientes relevantes
beta_x1 = coefs_poly['X1']
beta_x1_sq = coefs_poly.get('X1^2', 0)
beta_interaction = coefs_poly.get('X1*X2', 0)

for x1 in X1_values:
    effects = []
    for x2 in X2_values:
        # Efecto marginal = ∂y/∂x1 = β1 + 2β11*x1 + β12*x2
        effect = beta_x1 + 2 * beta_x1_sq * x1 + beta_interaction * x2
        effects.append(effect)
    
    print(f"{x1:10.2f}{effects[0]:15.4f}{effects[1]:15.4f}{effects[2]:15.4f}")

# Visualizar el efecto marginal como superficie
x1_range = np.linspace(X['X1'].min(), X['X1'].max(), 50)
x2_range = np.linspace(X['X2'].min(), X['X2'].max(), 50)
X1_grid, X2_grid = np.meshgrid(x1_range, x2_range)

# Calcular el efecto marginal para cada combinación de X1 y X2
marg_effect = beta_x1 + 2 * beta_x1_sq * X1_grid + beta_interaction * X2_grid

# Graficar superficie 3D del efecto marginal
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X1_grid, X2_grid, marg_effect, cmap='viridis', alpha=0.8)
ax.set_xlabel('X1')
ax.set_ylabel('X2')
ax.set_zlabel('Efecto Marginal de X1')
ax.set_title('Efecto Marginal de X1 como Función de X1 y X2')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
plt.show()


# Gestión de la Multicolinealidad en Modelos No Lineales

Los modelos con términos no lineales suelen enfrentar problemas de multicolinealidad, ya que los términos polinómicos y de interacción están correlacionados con las variables originales. A continuación, exploraremos estrategias para gestionar este problema.


In [ ]:

# 1. Centrado de Variables como Primera Estrategia
print("="*50)
print("ESTRATEGIA 1: CENTRADO DE VARIABLES")
print("="*50)

# Centrar las variables predictoras
X_centered = X_train - X_train.mean()
print("Media de las variables centradas:")
print(X_centered.mean())

# Crear términos polinómicos con variables centradas
X_centered_poly = crear_polinomios(X_centered, grado=2)
print("Variables en el modelo polinómico centrado:", X_centered_poly.columns.tolist())

# Ajustar modelo con variables centradas
modelo_centered = sm.OLS(y_train, sm.add_constant(X_centered_poly)).fit()
print(modelo_centered.summary())

# Calcular VIF para el modelo centrado
print("\nFactores de Inflación de la Varianza (VIF) para el modelo con variables centradas:")
vif_centered = calcular_vif(X_centered_poly)
print(vif_centered.sort_values('VIF', ascending=False))

# Comparar VIF antes y después del centrado
print("\nComparación de VIF antes y después del centrado:")
vif_comparison = pd.DataFrame({
    'Variable': vif_poly['Variable'],
    'VIF_Original': vif_poly['VIF'],
    'VIF_Centrado': [vif_centered.loc[vif_centered['Variable'] == var, 'VIF'].values[0] 
                      if var in vif_centered['Variable'].values else np.nan 
                      for var in vif_poly['Variable']]
})
print(vif_comparison)

# 2. Regularización como Segunda Estrategia
print("\n" + "="*50)
print("ESTRATEGIA 2: REGULARIZACIÓN (RIDGE)")
print("="*50)

# Preparar datos para Ridge
X_poly_array = PolynomialFeatures(degree=2, include_bias=False).fit_transform(X_train)
X_test_poly_array = PolynomialFeatures(degree=2, include_bias=False).fit_transform(X_test)

# Escalar características para mejorar la convergencia
scaler = StandardScaler()
X_poly_scaled = scaler.fit_transform(X_poly_array)
X_test_poly_scaled = scaler.transform(X_test_poly_array)

# Optimizar alfa mediante validación cruzada
alphas = np.logspace(-3, 3, 20)
ridge_cv = KFold(n_splits=5, shuffle=True, random_state=42)
ridge_mse = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    mse = -np.mean(cross_val_score(ridge, X_poly_scaled, y_train, 
                                 scoring='neg_mean_squared_error', cv=ridge_cv))
    ridge_mse.append(mse)

# Encontrar el mejor alfa
best_alpha_idx = np.argmin(ridge_mse)
best_alpha = alphas[best_alpha_idx]
print(f"Mejor valor de alfa: {best_alpha:.4f}")

# Ajustar modelo Ridge con el mejor alfa
ridge_model = Ridge(alpha=best_alpha)
ridge_model.fit(X_poly_scaled, y_train)

# Evaluar en conjunto de prueba
y_pred_ridge = ridge_model.predict(X_test_poly_scaled)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

print(f"Métricas en conjunto de prueba (Ridge):")
print(f"MSE: {mse_ridge:.4f}")
print(f"R²: {r2_ridge:.4f}")

# Comparar con OLS
print("\nComparación de MSE entre modelos:")
print(f"MSE OLS (sin regularización): {mse_poly2:.4f}")
print(f"MSE Ridge (con regularización): {mse_ridge:.4f}")
print(f"Mejora relativa: {(mse_poly2 - mse_ridge) / mse_poly2 * 100:.2f}%")

# Visualizar los coeficientes de Ridge vs OLS
poly_features = PolynomialFeatures(degree=2, include_bias=False)
poly_features.fit(X_train)
feature_names = poly_features.get_feature_names_out(X_train.columns)

# Obtener coeficientes de OLS
X_poly_with_const = sm.add_constant(X_train_poly2)
ols_coefs = modelo_poly2.params[1:]  # Excluir intercepto

# Obtener coeficientes de Ridge
ridge_coefs = ridge_model.coef_

# Crear DataFrame para comparación
coef_comparison = pd.DataFrame({
    'Feature': feature_names,
    'OLS': ols_coefs,
    'Ridge': ridge_coefs
})

plt.figure(figsize=(10, 6))
plt.bar(np.arange(len(feature_names)) - 0.2, ols_coefs, width=0.4, label='OLS')
plt.bar(np.arange(len(feature_names)) + 0.2, ridge_coefs, width=0.4, label='Ridge')
plt.xticks(np.arange(len(feature_names)), feature_names, rotation=90)
plt.legend()
plt.title('Comparación de Coeficientes: OLS vs Ridge')
plt.tight_layout()
plt.show()


# Validación de la Complejidad del Modelo No Lineal

Un desafío importante al trabajar con modelos no lineales es determinar el nivel de complejidad apropiado. A continuación, exploraremos métodos para validar y seleccionar el grado óptimo de complejidad para nuestros modelos.


In [ ]:

# Evaluar modelos con diferentes grados de complejidad
print("="*50)
print("EVALUACIÓN DE DIFERENTES GRADOS DE COMPLEJIDAD")
print("="*50)

# Probar diferentes grados polinómicos
grados = range(1, 5)  # Grados 1 al 4
resultados = []

for grado in grados:
    # Evaluar con validación cruzada
    resultado = evaluar_modelo_cv(X_train, y_train, grado=grado, cv=5)
    resultado['grado'] = grado
    resultados.append(resultado)
    
    print(f"\nResultados para Grado {grado}:")
    print(f"R² Train: {resultado['r2_train']:.4f}")
    print(f"MSE Train: {resultado['mse_train']:.4f}")
    print(f"MSE CV (media ± desv. est.): {resultado['cv_mean']:.4f} ± {resultado['cv_std']:.4f}")

# Visualizar los resultados
mse_train = [res['mse_train'] for res in resultados]
mse_cv = [res['cv_mean'] for res in resultados]
mse_cv_std = [res['cv_std'] for res in resultados]

plt.figure(figsize=(10, 6))
plt.errorbar(grados, mse_cv, yerr=mse_cv_std, fmt='-o', label='MSE Validación Cruzada')
plt.plot(grados, mse_train, '-s', label='MSE Entrenamiento')
plt.xlabel('Grado del Polinomio')
plt.ylabel('Error Cuadrático Medio (MSE)')
plt.title('MSE vs. Grado del Polinomio')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Encontrar el grado óptimo
mejor_grado_idx = np.argmin([res['cv_mean'] for res in resultados])
mejor_grado = grados[mejor_grado_idx]
print(f"\nEl grado óptimo según validación cruzada es: {mejor_grado}")

# Implementar modelo final con el grado óptimo
print("\n" + "="*50)
print(f"MODELO FINAL: GRADO {mejor_grado}")
print("="*50)

# Crear características con el grado óptimo
X_final_poly = crear_polinomios(X_train, grado=mejor_grado)
print(f"Características en el modelo final: {X_final_poly.columns.tolist()}")

# Ajustar modelo final
modelo_final = sm.OLS(y_train, sm.add_constant(X_final_poly)).fit()
print(modelo_final.summary())

# Evaluar en conjunto de prueba
X_test_final = crear_polinomios(X_test, grado=mejor_grado)
y_pred_final = modelo_final.predict(sm.add_constant(X_test_final))
mse_final = mean_squared_error(y_test, y_pred_final)
r2_final = r2_score(y_test, y_pred_final)

print(f"\nMétricas finales en conjunto de prueba:")
print(f"MSE: {mse_final:.4f}")
print(f"R²: {r2_final:.4f}")

# Comparar los resultados de todos los modelos
modelos = ['Lineal', 'Interacción', 'Polinómico (Grado 2)', 'Ridge', f'Final (Grado {mejor_grado})']
mse_valores = [mse_lineal, mse_inter, mse_poly2, mse_ridge, mse_final]
r2_valores = [r2_lineal, r2_inter, r2_poly2, r2_ridge, r2_final]

# Crear tabla comparativa
comparacion = pd.DataFrame({
    'Modelo': modelos,
    'MSE': mse_valores,
    'R²': r2_valores
})
print("\nComparación de todos los modelos:")
print(comparacion)

# Visualizar comparación
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.bar(modelos, mse_valores)
plt.ylabel('MSE')
plt.title('Comparación de MSE')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.bar(modelos, r2_valores)
plt.ylabel('R²')
plt.title('Comparación de R²')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# Análisis de Residuos para Modelos No Lineales

El análisis de residuos es especialmente importante en modelos no lineales para verificar si los términos no lineales capturan adecuadamente las relaciones en los datos o si existen patrones no modelados.


In [ ]:

# Obtener residuos para diferentes modelos
residuos_lineal = y_test - y_pred_lineal
residuos_poly2 = y_test - y_pred_poly2
residuos_final = y_test - y_pred_final

# Función para crear gráficos de diagnóstico de residuos
def plot_residuos_diagnostico(y_true, y_pred, titulo):
    residuos = y_true - y_pred
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(titulo, fontsize=16)
    
    # 1. Residuos vs Valores Ajustados
    axes[0, 0].scatter(y_pred, residuos, alpha=0.5)
    axes[0, 0].axhline(y=0, color='r', linestyle='-')
    axes[0, 0].set_xlabel('Valores Ajustados')
    axes[0, 0].set_ylabel('Residuos')
    axes[0, 0].set_title('Residuos vs Valores Ajustados')
    
    # 2. Histograma de Residuos
    axes[0, 1].hist(residuos, bins=20, edgecolor='black')
    axes[0, 1].set_xlabel('Residuos')
    axes[0, 1].set_ylabel('Frecuencia')
    axes[0, 1].set_title('Histograma de Residuos')
    
    # 3. Q-Q Plot
    from scipy import stats
    qq = stats.probplot(residuos, dist="norm", plot=axes[1, 0])
    axes[1, 0].set_title('Q-Q Plot')
    
    # 4. Residuos Estandarizados vs Valores Ajustados
    std_residuos = (residuos - residuos.mean()) / residuos.std()
    axes[1, 1].scatter(y_pred, std_residuos, alpha=0.5)
    axes[1, 1].axhline(y=0, color='r', linestyle='-')
    axes[1, 1].axhline(y=2, color='r', linestyle='--')
    axes[1, 1].axhline(y=-2, color='r', linestyle='--')
    axes[1, 1].set_xlabel('Valores Ajustados')
    axes[1, 1].set_ylabel('Residuos Estandarizados')
    axes[1, 1].set_title('Residuos Estandarizados vs Valores Ajustados')
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    return fig

# Generar gráficos de diagnóstico para cada modelo
print("Gráficos de diagnóstico para el modelo lineal:")
fig_lineal = plot_residuos_diagnostico(y_test, y_pred_lineal, "Diagnóstico de Residuos - Modelo Lineal")
plt.show()

print("\nGráficos de diagnóstico para el modelo polinómico (grado 2):")
fig_poly2 = plot_residuos_diagnostico(y_test, y_pred_poly2, "Diagnóstico de Residuos - Modelo Polinómico (Grado 2)")
plt.show()

print("\nGráficos de diagnóstico para el modelo final:")
fig_final = plot_residuos_diagnostico(y_test, y_pred_final, f"Diagnóstico de Residuos - Modelo Final (Grado {mejor_grado})")
plt.show()

# Realizar pruebas estadísticas sobre los residuos
from scipy import stats

# Prueba de normalidad (Shapiro-Wilk)
print("\n" + "="*50)
print("PRUEBAS DE NORMALIDAD EN RESIDUOS (SHAPIRO-WILK)")
print("="*50)
alpha = 0.05

_, p_value_lineal = stats.shapiro(residuos_lineal)
print(f"Modelo Lineal: p-value = {p_value_lineal:.4f}")
print(f"{'Los residuos son normales' if p_value_lineal > alpha else 'Los residuos NO son normales'}")

_, p_value_poly2 = stats.shapiro(residuos_poly2)
print(f"Modelo Polinómico (Grado 2): p-value = {p_value_poly2:.4f}")
print(f"{'Los residuos son normales' if p_value_poly2 > alpha else 'Los residuos NO son normales'}")

_, p_value_final = stats.shapiro(residuos_final)
print(f"Modelo Final (Grado {mejor_grado}): p-value = {p_value_final:.4f}")
print(f"{'Los residuos son normales' if p_value_final > alpha else 'Los residuos NO son normales'}")

# Prueba de homocedasticidad (Breusch-Pagan)
import statsmodels.stats.diagnostic as diag

print("\n" + "="*50)
print("PRUEBAS DE HOMOCEDASTICIDAD EN RESIDUOS (BREUSCH-PAGAN)")
print("="*50)

# Para el modelo lineal
bp_test_lineal = diag.het_breuschpagan(residuos_lineal, sm.add_constant(y_pred_lineal.reshape(-1, 1)))
print(f"Modelo Lineal: p-value = {bp_test_lineal[1]:.4f}")
print(f"{'Varianza constante (homocedasticidad)' if bp_test_lineal[1] > alpha else 'Varianza NO constante (heterocedasticidad)'}")

# Para el modelo polinómico grado 2
bp_test_poly2 = diag.het_breuschpagan(residuos_poly2, sm.add_constant(y_pred_poly2.reshape(-1, 1)))
print(f"Modelo Polinómico (Grado 2): p-value = {bp_test_poly2[1]:.4f}")
print(f"{'Varianza constante (homocedasticidad)' if bp_test_poly2[1] > alpha else 'Varianza NO constante (heterocedasticidad)'}")

# Para el modelo final
bp_test_final = diag.het_breuschpagan(residuos_final, sm.add_constant(y_pred_final.reshape(-1, 1)))
print(f"Modelo Final (Grado {mejor_grado}): p-value = {bp_test_final[1]:.4f}")
print(f"{'Varianza constante (homocedasticidad)' if bp_test_final[1] > alpha else 'Varianza NO constante (heterocedasticidad)'}")


# Caso de Estudio con Datos Reales

Para consolidar los conceptos aprendidos, aplicaremos las técnicas de modelación no lineal a un conjunto de datos reales. Utilizaremos el dataset de precios de viviendas de Boston, un dataset clásico que contiene variables que podrían tener relaciones no lineales con el precio de las viviendas.


In [ ]:

# Importar el dataset de Boston Housing
from sklearn.datasets import load_boston
import warnings
warnings.filterwarnings('ignore')  # Ignorar advertencias de obsolescencia

# Cargar datos
try:
    boston = load_boston()
    X_boston = pd.DataFrame(boston.data, columns=boston.feature_names)
    y_boston = pd.Series(boston.target, name='PRICE')
    print("Dataset de Boston Housing cargado correctamente.")
    
    # En caso de que el dataset no esté disponible en versiones futuras de scikit-learn
except Exception as e:
    print(f"Error al cargar el dataset de Boston Housing: {e}")
    print("Generando datos sintéticos como alternativa...")
    
    # Generar datos sintéticos como alternativa
    np.random.seed(42)
    n = 500
    
    # Crear predictores
    X_boston = pd.DataFrame({
        'CRIM': np.random.exponential(0.5, n),        # Tasa de criminalidad
        'ZN': np.random.uniform(0, 100, n),           # % suelo residencial
        'INDUS': np.random.uniform(0, 30, n),         # % suelo industrial
        'CHAS': np.random.binomial(1, 0.07, n),       # Proximidad al río Charles
        'NOX': np.random.uniform(0.4, 0.9, n),        # Concentración de óxido nítrico
        'RM': np.random.normal(6.5, 0.7, n),          # Promedio de habitaciones
        'AGE': np.random.uniform(0, 100, n),          # % edificios anteriores a 1940
        'DIS': np.random.uniform(1, 12, n),           # Distancias ponderadas
        'RAD': np.random.choice(range(1, 25), n),     # Accesibilidad a carreteras
        'TAX': np.random.uniform(100, 700, n),        # Tasa de impuestos
        'PTRATIO': np.random.uniform(12, 22, n),      # Ratio alumno-profesor
        'B': np.random.uniform(0, 400, n),            # Proporción de residentes negros
        'LSTAT': np.random.uniform(1, 40, n)          # % de población de estatus bajo
    })
    
    # Crear variable respuesta (precio) con relaciones no lineales
    y_boston = 10 + 0.1 * X_boston['ZN'] - 0.2 * X_boston['INDUS'] + 3 * X_boston['RM'] \
            + 2.5 * X_boston['RM']**2 - 0.5 * X_boston['NOX']**2 \
            - 0.8 * X_boston['LSTAT'] - 0.2 * X_boston['LSTAT']**2 \
            + 0.01 * X_boston['RM'] * X_boston['DIS'] \
            + np.random.normal(0, 3, n)
    
    y_boston = pd.Series(y_boston, name='PRICE')
    print("Datos sintéticos generados correctamente.")

# Explorar los datos
print("\nDimensiones del dataset:")
print(f"X: {X_boston.shape}, y: {y_boston.shape}")

print("\nEstadísticas descriptivas:")
print(X_boston.describe())

print("\nPrimeras filas del dataset:")
print(pd.concat([X_boston.head(), y_boston.head()], axis=1))

# Visualizar relaciones bivariadas entre variables seleccionadas y precio
variables_interes = ['RM', 'LSTAT', 'DIS', 'NOX']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, var in enumerate(variables_interes):
    sns.regplot(x=X_boston[var], y=y_boston, ax=axes[i], 
                scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'})
    axes[i].set_title(f'Relación entre {var} y PRICE')
    
plt.tight_layout()
plt.show()

# Dividir los datos en entrenamiento y prueba
X_train_boston, X_test_boston, y_train_boston, y_test_boston = train_test_split(
    X_boston, y_boston, test_size=0.25, random_state=42
)

# Modelo lineal como base de comparación
modelo_lineal_boston = sm.OLS(y_train_boston, sm.add_constant(X_train_boston)).fit()
print("\nResultados del modelo lineal base:")
print(modelo_lineal_boston.summary().tables[1])  # Solo mostrar tabla de coeficientes

# Evaluar modelo lineal
y_pred_lineal_boston = modelo_lineal_boston.predict(sm.add_constant(X_test_boston))
mse_lineal_boston = mean_squared_error(y_test_boston, y_pred_lineal_boston)
r2_lineal_boston = r2_score(y_test_boston, y_pred_lineal_boston)

print(f"\nMétricas del modelo lineal en test:")
print(f"MSE: {mse_lineal_boston:.4f}")
print(f"R²: {r2_lineal_boston:.4f}")

# Implementar modelo con términos cuadráticos para variables seleccionadas
print("\n" + "="*50)
print("MODELO CON TÉRMINOS CUADRÁTICOS PARA VARIABLES SELECCIONADAS")
print("="*50)

# Seleccionar variables para términos cuadráticos basados en visualización
vars_cuadraticas = ['RM', 'LSTAT', 'DIS', 'NOX']

# Crear DataFrame con términos cuadráticos
X_train_quad = X_train_boston.copy()
for var in vars_cuadraticas:
    X_train_quad[f'{var}^2'] = X_train_boston[var]**2

# Crear términos de interacción para variables seleccionadas
X_train_quad['RM*LSTAT'] = X_train_boston['RM'] * X_train_boston['LSTAT']
X_train_quad['RM*DIS'] = X_train_boston['RM'] * X_train_boston['DIS']

# Ajustar modelo
modelo_quad_boston = sm.OLS(y_train_boston, sm.add_constant(X_train_quad)).fit()
print("Resultados del modelo con términos cuadráticos:")
print(modelo_quad_boston.summary().tables[1])  # Solo mostrar tabla de coeficientes

# Evaluar modelo en test
X_test_quad = X_test_boston.copy()
for var in vars_cuadraticas:
    X_test_quad[f'{var}^2'] = X_test_boston[var]**2
X_test_quad['RM*LSTAT'] = X_test_boston['RM'] * X_test_boston['LSTAT']
X_test_quad['RM*DIS'] = X_test_boston['RM'] * X_test_boston['DIS']

y_pred_quad_boston = modelo_quad_boston.predict(sm.add_constant(X_test_quad))
mse_quad_boston = mean_squared_error(y_test_boston, y_pred_quad_boston)
r2_quad_boston = r2_score(y_test_boston, y_pred_quad_boston)

print(f"\nMétricas del modelo con términos cuadráticos en test:")
print(f"MSE: {mse_quad_boston:.4f}")
print(f"R²: {r2_quad_boston:.4f}")

# Comparar modelos
mejora_porcentual = (mse_lineal_boston - mse_quad_boston) / mse_lineal_boston * 100
print(f"\nMejora porcentual en MSE: {mejora_porcentual:.2f}%")

# Visualizar la interpretación de efectos no lineales
vars_efecto = ['RM', 'LSTAT']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, var in enumerate(vars_efecto):
    # Crear rango de valores para la variable
    x_range = np.linspace(X_boston[var].min(), X_boston[var].max(), 100)
    
    # Crear datos para predicción
    X_pred = X_test_boston.mean().to_dict()  # Valores medios para todas las variables
    y_preds = []
    
    for x_val in x_range:
        X_pred_df = pd.DataFrame([X_pred])
        X_pred_df[var] = x_val
        
        # Crear términos cuadráticos e interacciones
        X_pred_quad = X_pred_df.copy()
        for v in vars_cuadraticas:
            X_pred_quad[f'{v}^2'] = X_pred_df[v]**2
        X_pred_quad['RM*LSTAT'] = X_pred_df['RM'] * X_pred_df['LSTAT']
        X_pred_quad['RM*DIS'] = X_pred_df['RM'] * X_pred_df['DIS']
        
        # Hacer predicción
        y_pred = modelo_quad_boston.predict(sm.add_constant(X_pred_quad))
        y_preds.append(y_pred[0])
    
    # Graficar la relación no lineal
    axes[i].scatter(X_boston[var], y_boston, alpha=0.2, label='Datos')
    axes[i].plot(x_range, y_preds, 'r-', linewidth=2, label='Efecto no lineal')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('PRICE')
    axes[i].set_title(f'Efecto no lineal de {var} sobre PRICE')
    axes[i].legend()

plt.tight_layout()
plt.show()

# Calcular y visualizar el efecto marginal
print("\n" + "="*50)
print("EFECTO MARGINAL DE VARIABLES SELECCIONADAS")
print("="*50)

# Extraer coeficientes relevantes para RM
beta_rm = modelo_quad_boston.params[modelo_quad_boston.params.index == 'RM'][0]
beta_rm_sq = modelo_quad_boston.params[modelo_quad_boston.params.index == 'RM^2'][0]
beta_rm_lstat = modelo_quad_boston.params[modelo_quad_boston.params.index == 'RM*LSTAT'][0]
beta_rm_dis = modelo_quad_boston.params[modelo_quad_boston.params.index == 'RM*DIS'][0]

# Extraer coeficientes relevantes para LSTAT
beta_lstat = modelo_quad_boston.params[modelo_quad_boston.params.index == 'LSTAT'][0]
beta_lstat_sq = modelo_quad_boston.params[modelo_quad_boston.params.index == 'LSTAT^2'][0]

print("Efecto marginal de RM (habitaciones promedio):")
print(f"∂PRICE/∂RM = {beta_rm:.4f} + 2*{beta_rm_sq:.4f}*RM + {beta_rm_lstat:.4f}*LSTAT + {beta_rm_dis:.4f}*DIS")

print("\nEfecto marginal de LSTAT (% población de estatus bajo):")
print(f"∂PRICE/∂LSTAT = {beta_lstat:.4f} + 2*{beta_lstat_sq:.4f}*LSTAT + {beta_rm_lstat:.4f}*RM")

# Visualizar el efecto marginal de RM para diferentes valores de LSTAT
# Crear grid para RM y LSTAT
rm_range = np.linspace(X_boston['RM'].min(), X_boston['RM'].max(), 50)
lstat_range = np.linspace(X_boston['LSTAT'].min(), X_boston['LSTAT'].max(), 50)
rm_grid, lstat_grid = np.meshgrid(rm_range, lstat_range)

# Calcular el efecto marginal
dis_mean = X_boston['DIS'].mean()  # Valor fijo para DIS
marg_effect_rm = beta_rm + 2 * beta_rm_sq * rm_grid + beta_rm_lstat * lstat_grid + beta_rm_dis * dis_mean

# Graficar superficie del efecto marginal
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(rm_grid, lstat_grid, marg_effect_rm, cmap='viridis', alpha=0.8)
ax.set_xlabel('RM (habitaciones promedio)')
ax.set_ylabel('LSTAT (% población estatus bajo)')
ax.set_zlabel('Efecto Marginal de RM')
ax.set_title('Efecto Marginal de RM como Función de RM y LSTAT')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
plt.show()


# Ejercicios Prácticos

A continuación se proponen varios ejercicios para reforzar los conceptos aprendidos sobre extensiones no lineales en modelos lineales.

### Ejercicio 1: Términos Polinómicos

Utiliza el siguiente código para generar un conjunto de datos con relación cuadrática:

```python
np.random.seed(123)
n = 200
X1 = np.random.uniform(-5, 5, n)
y = 2 + 0.5 * X1 - 1.5 * X1**2 + np.random.normal(0, 1, n)
X = pd.DataFrame({'X1': X1})
```

1. Visualiza la relación entre X1 e y mediante un gráfico de dispersión.
2. Ajusta un modelo lineal simple.
3. Ajusta un modelo polinómico de grado 2.
4. Ajusta un modelo polinómico de grado 3.
5. Compara los tres modelos en términos de R² y MSE.
6. ¿Cuál es el grado óptimo para este conjunto de datos? ¿Por qué?

### Ejercicio 2: Interacciones

Utiliza el siguiente código para generar un conjunto de datos con interacción:

```python
np.random.seed(456)
n = 150
X1 = np.random.uniform(-3, 3, n)
X2 = np.random.uniform(-3, 3, n)
y = 1 - 0.5 * X1 + 0.8 * X2 + 1.2 * X1 * X2 + np.random.normal(0, 0.5, n)
X = pd.DataFrame({'X1': X1, 'X2': X2})
```

1. Ajusta un modelo sin interacción (solo términos lineales).
2. Ajusta un modelo con interacción.
3. Visualiza cómo cambia el efecto de X1 sobre y para diferentes valores de X2.
4. Calcula e interpreta el efecto marginal de X1 para X2 = -2, X2 = 0 y X2 = 2.

### Ejercicio 3: Multicolinealidad y Regularización

Utiliza el siguiente código para generar un conjunto de datos con alta multicolinealidad:

```python
np.random.seed(789)
n = 100
X1 = np.random.normal(0, 1, n)
X2 = 0.8 * X1 + np.random.normal(0, 0.2, n)  # X2 correlacionada con X1
X3 = np.random.normal(0, 1, n)
y = 2 + 1.5 * X1 - 0.5 * X2 + 0.8 * X3 + 1.2 * X1**2 - 0.7 * X2**2 + np.random.normal(0, 1, n)
X = pd.DataFrame({'X1': X1, 'X2': X2, 'X3': X3})
```

1. Calcula la matriz de correlación entre X1, X2 y X3.
2. Ajusta un modelo polinómico de grado 2 para todas las variables.
3. Calcula e interpreta los VIF para las variables en el modelo.
4. Implementa regularización Ridge para mitigar la multicolinealidad.
5. Compara los coeficientes del modelo OLS con los del modelo Ridge.
6. ¿Cómo afecta la multicolinealidad a la interpretación de los efectos no lineales?

### Ejercicio 4: Validación de la Complejidad

Utilizando cualquiera de los conjuntos de datos anteriores:

1. Implementa validación cruzada para seleccionar el grado óptimo del polinomio.
2. Evalúa modelos desde grado 1 hasta grado 5.
3. Visualiza el MSE de entrenamiento y validación para cada grado.
4. Identifica signos de sobreajuste.
5. Selecciona y justifica el modelo final.


# Conclusiones

En este taller hemos explorado las extensiones no lineales para modelos lineales, centrándonos en términos polinómicos e interacciones. Los principales conceptos y aprendizajes incluyen:

1. **Tipos de Extensiones No Lineales**:
   - Los términos polinómicos permiten modelar relaciones curvilíneas entre predictores y la variable respuesta.
   - Las interacciones capturan cómo el efecto de una variable depende del valor de otra.
   - Estas extensiones amplían significativamente la flexibilidad de los modelos lineales.

2. **Interpretación de Coeficientes**:
   - La interpretación se vuelve más compleja en modelos no lineales.
   - El efecto marginal deja de ser constante y depende de los valores de las variables.
   - La visualización de efectos es especialmente útil para una correcta interpretación.

3. **Multicolinealidad Inducida**:
   - Los términos no lineales suelen introducir multicolinealidad.
   - El centrado de variables y la regularización son técnicas efectivas para mitigar este problema.
   - Los VIF son herramientas diagnósticas importantes para detectar multicolinealidad.

4. **Validación de la Complejidad**:
   - Es fundamental validar el nivel de complejidad para evitar el sobreajuste.
   - La validación cruzada proporciona una estimación más realista del error de generalización.
   - El equilibrio entre complejidad y capacidad predictiva es esencial.

5. **Análisis de Residuos**:
   - El análisis de residuos permite verificar si los términos no lineales capturan adecuadamente las relaciones.
   - Los patrones en los residuos pueden sugerir transformaciones o términos adicionales necesarios.

La implementación adecuada de extensiones no lineales puede mejorar significativamente el poder predictivo y la interpretabilidad de nuestros modelos, siempre que se manejen correctamente los desafíos que conllevan, como la multicolinealidad y el riesgo de sobreajuste.


# Referencias

1. Fox, J. (2015). *Applied Regression Analysis and Generalized Linear Models* (3rd ed.). SAGE Publications.

2. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2013). *An Introduction to Statistical Learning: with Applications in R*. Springer.

3. Kutner, M. H., Nachtsheim, C. J., Neter, J., & Li, W. (2005). *Applied Linear Statistical Models* (5th ed.). McGraw-Hill/Irwin.

4. Harrell, F. E. (2015). *Regression Modeling Strategies: With Applications to Linear Models, Logistic and Ordinal Regression, and Survival Analysis* (2nd ed.). Springer.

5. Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning: Data Mining, Inference, and Prediction* (2nd ed.). Springer.

6. Ryan, T. P. (2008). *Modern Regression Methods* (2nd ed.). Wiley.

7. Faraway, J. J. (2016). *Linear Models with R* (2nd ed.). CRC Press.
